<i18n value="931bf77d-810b-4930-b45c-b00c184029a0"/>

# SQL UDF 및 제어 흐름

Databricks는 DBR 9.1부터 SQL에 기본적으로 등록된 사용자 정의 함수(UDF)에 대한 지원을 추가했습니다.

이 기능을 통해 사용자는 사용자 정의 SQL 로직 조합을 데이터베이스에 함수로 등록하여 Databricks에서 SQL을 실행할 수 있는 모든 곳에서 이러한 메서드를 재사용할 수 있습니다. 이러한 함수는 Spark SQL을 직접 활용하여 사용자 정의 로직을 대규모 데이터세트에 적용할 때 Spark의 모든 최적화를 유지합니다.

이 노트북에서는 먼저 이러한 메서드에 대한 간단한 소개를 한 다음, 이 로직을 **`CASE`** / **`WHEN`** 절과 결합하여 재사용 가능한 사용자 정의 제어 흐름 로직을 제공하는 방법을 살펴보겠습니다.

## 학습 목표
이 수업을 마치면 다음을 수행할 수 있어야 합니다.
* SQL UDF 정의 및 등록
* SQL UDF 공유에 사용되는 보안 모델 설명
* SQL 코드에서 **`CASE`** / **`WHEN`** 문 사용
* 사용자 지정 제어 흐름을 위해 SQL UDF에서 **`CASE`** / **`WHEN`** 문 활용

<i18n value="df80ac46-fb12-44ed-bb37-dcc5a4d73d4a"/>

## 설정
다음 셀을 실행하여 환경을 설정하세요.

In [0]:
%run ../Includes/Classroom-Setup-04.8

Python interpreter will be restarted.
Python interpreter will be restarted.


Resetting the learning environment:
| No action taken

Skipping install of existing datasets to "dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02"

Validating the locally installed datasets:
| listing local files...(4 seconds)
| validation completed...(4 seconds total)

Creating & using the schema "3dt005_nuk5_da_dewd" in the catalog "hive_metastore"...(1 seconds)

Predefined tables in "3dt005_nuk5_da_dewd":
| -none-

Predefined paths variables:
| DA.paths.working_dir: dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineering-with-databricks
| DA.paths.user_db:     dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineering-with-databricks/database.db
| DA.paths.datasets:    dbfs:/mnt/dbacademy-datasets/data-engineering-with-databricks/v02
| DA.paths.checkpoints: dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineering-with-databricks/_checkpoints

Setup completed (6 seconds)


<i18n value="f4fec594-3cd7-43c9-b88e-3ccd3a99c6be"/>

## 간단한 데이터셋 만들기

이 노트북에서는 임시 뷰로 등록된 다음 데이터셋을 살펴보겠습니다.

In [0]:
CREATE OR REPLACE TEMPORARY VIEW foods(food) AS VALUES
("beef"),
("beans"),
("potatoes"),
("bread");

SELECT * FROM foods

food
beef
beans
potatoes
bread


<i18n value="65577a77-c917-441c-895b-8ba146c837ff"/>


## SQL UDFs
SQL UDF에는 최소한 함수 이름, 선택적 매개변수, 반환될 형식, 그리고 몇 가지 사용자 지정 로직이 필요합니다.

아래의 **yelling**이라는 간단한 함수는 **text**라는 매개변수 하나를 받습니다. 이 함수는 모두 대문자로 작성되고 끝에 느낌표 세 개가 추가된 문자열을 반환합니다.

In [0]:
CREATE OR REPLACE FUNCTION yelling(text STRING)
RETURNS STRING
RETURN concat(upper(text), "!!!")

<i18n value="4cffc92d-3133-45ba-97c8-b0bc4c9e419b"/>

이 함수는 Spark 처리 엔진 내에서 열의 모든 값에 병렬 방식으로 적용됩니다. SQL UDF는 Databricks에서 실행되도록 최적화된 사용자 지정 로직을 정의하는 효율적인 방법입니다.

In [0]:
SELECT yelling(food) FROM foods

hive_metastore.3dt005_nuk5_da_dewd.yelling(food)
BEEF!!!
BEANS!!!
POTATOES!!!
BREAD!!!


<i18n value="e1749d08-2186-4e1c-9214-18c8199388af"/>

## SQL UDF의 범위 및 권한

SQL UDF는 실행 환경(노트북, DBSQL 쿼리, 작업 등) 간에 유지됩니다.

함수를 설명하여 함수가 등록된 위치, 예상 입력 및 반환되는 내용에 대한 기본 정보를 확인할 수 있습니다.

In [0]:
DESCRIBE FUNCTION yelling

function_desc
Function: hive_metastore.3dt005_nuk5_da_dewd.yelling
Type: SCALAR
Input: text STRING
Returns: STRING


<i18n value="6a6eb6c6-ffc8-49d9-a39a-a5e1f6c230af"/>

extended를 설명하면 더 많은 정보를 얻을 수 있습니다.

함수 설명 하단의 **`Body`** 필드는 함수 자체에서 사용된 SQL 로직을 보여줍니다.

In [0]:
DESCRIBE FUNCTION EXTENDED yelling

function_desc
Function: hive_metastore.3dt005_nuk5_da_dewd.yelling
Type: SCALAR
Input: text STRING
Returns: STRING
Deterministic: true
Data Access: CONTAINS SQL
Configs: spark.sql.hive.convertCTAS=true
spark.sql.legacy.createHiveTableByDefault=false
spark.sql.parquet.compression.codec=snappy
spark.sql.sources.commitProtocolClass=com.databricks.sql.transaction.directory.DirectoryAtomicCommitProtocol


<i18n value="a31a4ad1-5608-4bfb-aae4-a411fe460385"/>

SQL UDF는 메타스토어에 객체로 존재하며 데이터베이스, 테이블 또는 뷰와 동일한 테이블 ACL에 의해 관리됩니다.

SQL UDF를 사용하려면 사용자에게 해당 함수에 대한 **`USAGE`** 및 **`SELECT`** 권한이 있어야 합니다.

<i18n value="155c70b7-ed5e-47d2-9832-963aa18f3869"/>

## CASE/WHEN

표준 SQL 구문 구조 **`CASE`** / **`WHEN`**을 사용하면 테이블 내용에 따라 다른 결과를 갖는 여러 조건문을 평가할 수 있습니다.

다시 말하지만, 모든 것은 Spark에서 기본적으로 평가되므로 병렬 실행에 최적화되어 있습니다.

In [0]:
SELECT *,
  CASE 
    WHEN food = "beans" THEN "I love beans"
    WHEN food = "potatoes" THEN "My favorite vegetable is potatoes"
    WHEN food <> "beef" THEN concat("Do you have any good recipes for ", food ,"?")
    ELSE concat("I don't eat ", food)
  END
FROM foods

food,"CASE WHEN (food = beans) THEN I love beans WHEN (food = potatoes) THEN My favorite vegetable is potatoes WHEN (NOT (food = beef)) THEN concat(Do you have any good recipes for , food, ?) ELSE concat(I don't eat , food) END"
beef,I don't eat beef
beans,I love beans
potatoes,My favorite vegetable is potatoes
bread,Do you have any good recipes for bread?


<i18n value="50bc0847-94d2-4167-befe-66e42b287ad0"/>

## 간단한 제어 흐름 함수

SQL UDF를 **`CASE`** / **`WHEN`** 절 형태의 제어 흐름과 결합하면 SQL 워크로드 내에서 제어 흐름 실행을 최적화할 수 있습니다.

여기서는 SQL을 실행할 수 있는 모든 곳에서 재사용 가능한 함수로 이전 로직을 래핑하는 방법을 보여줍니다.

In [0]:
CREATE FUNCTION foods_i_like(food STRING)
RETURNS STRING
RETURN CASE 
  WHEN food = "beans" THEN "I love beans"
  WHEN food = "potatoes" THEN "My favorite vegetable is potatoes"
  WHEN food <> "beef" THEN concat("Do you have any good recipes for ", food ,"?")
  ELSE concat("I don't eat ", food)
END;

<i18n value="05cb00cc-097c-4607-8738-ab4353536dda"/>

이 방법을 데이터에 적용하면 원하는 결과를 얻을 수 있습니다.

In [0]:
SELECT foods_i_like(food) FROM foods

hive_metastore.3dt005_nuk5_da_dewd.foods_i_like(food)
I don't eat beef
I love beans
My favorite vegetable is potatoes
Do you have any good recipes for bread?


<i18n value="24ee3267-9ddb-4cf5-9081-273502f5252a"/>

여기에 제공된 예제는 간단한 문자열 메서드이지만, 동일한 기본 원칙을 사용하여 Spark SQL에서 네이티브 실행을 위한 사용자 지정 계산 및 로직을 추가할 수 있습니다.

특히 정의된 프로시저나 사용자 지정 수식이 많은 시스템에서 사용자를 마이그레이션하는 기업의 경우, SQL UDF를 사용하면 소수의 사용자가 일반적인 보고 및 분석 쿼리에 필요한 복잡한 로직을 정의할 수 있습니다.

<i18n value="9405ddea-5fb0-4168-9fd2-2b462d5809d9"/>

이 레슨과 관련된 표와 파일을 삭제하려면 다음 셀을 실행하세요.

In [0]:
%python
DA.cleanup()

Resetting the learning environment:
| dropping the schema "3dt005_nuk5_da_dewd"...(1 seconds)
| removing the working directory "dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineering-with-databricks"...(0 seconds)

Validating the locally installed datasets:
| listing local files...(3 seconds)
| validation completed...(3 seconds total)
